# 🔬 Lens Agent — Draft

Lens is the experiment runner in the **Seesaw** pipeline. It takes a Research Plan from Scout,
runs mechanistic interpretability experiments, and produces a Results Bundle for Quill to critique.

## Pipeline position

```
Scout → [Research Plan] → Lens → [ExperimentBundle] → Quill → [CritiqueReport]
```

## Tier 1 Tools (this notebook)

| Tool | What it answers |
|---|---|
| `logit_lens` | Where does the model "decide"? |
| `attention_pattern` | What does each head attend to? |
| `direct_logit_attribution` | Which heads/MLPs contribute to the final prediction? |
| `ablation` | Is this component causally necessary? |
| `activation_patching` | Which layer+position carries the critical information? |

## Architecture
- **LangGraph** `StateGraph` — deterministic workflow, not ReAct
- **Claude** — result interpretation + follow-up decisions only
- **TransformerLens** — all experiment execution
- **matplotlib** — visualisations

## 1. Setup

In [ ]:
%pip install -q \
    transformer_lens \
    langchain-anthropic \
    langgraph \
    matplotlib \
    numpy \
    python-dotenv

In [ ]:
import os
import json
import concurrent.futures
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # non-interactive backend — saves to file

import torch
import torch.nn.functional as F
from transformer_lens import HookedTransformer

from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict

load_dotenv()
print("✅ Imports OK")

In [ ]:
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

LENS_ROOT   = Path(".")
OUTPUTS_DIR = LENS_ROOT / "outputs"
PLOTS_DIR   = OUTPUTS_DIR / "plots"
OUTPUTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

llm = ChatAnthropic(
    model="claude-sonnet-4-5",
    api_key=ANTHROPIC_API_KEY,
    temperature=0,
    max_tokens=2_000,
)

print(f"API key   : {'✅ set' if ANTHROPIC_API_KEY else '❌ missing'}")
print(f"Plots dir : {PLOTS_DIR.resolve()}")

## 2. Schemas

In [ ]:
@dataclass
class ExperimentSpec:
    name: str
    tool: str
    model_name: str
    prompts: list[str]
    what_to_measure: str
    hypothesis_tested: str
    expected_outcome: str
    tool_kwargs: dict = field(default_factory=dict)  # extra tool-specific params


@dataclass
class ExperimentResult:
    name: str
    tool: str
    model_name: str
    prompts: list[str]
    summary: str = ""
    plot_paths: list[Path] = field(default_factory=list)
    data: dict = field(default_factory=dict)
    status: Literal["success", "failed", "timeout"] = "success"
    error: str | None = None


@dataclass
class ExperimentBundle:
    research_question: str
    model_name: str
    results: list[ExperimentResult]

    @property
    def successful(self):
        return [r for r in self.results if r.status == "success"]

    @property
    def failed(self):
        return [r for r in self.results if r.status != "success"]


print("✅ Schemas defined")

## 3. Model Session

In [ ]:
_MODEL_CACHE: dict[str, HookedTransformer] = {}


def get_model(model_name: str) -> HookedTransformer:
    if model_name not in _MODEL_CACHE:
        print(f"⏳ Loading {model_name}...")
        model = HookedTransformer.from_pretrained(
            model_name,
            center_writing_weights=True,
            center_unembed=True,
            fold_ln=True,
            refactor_factored_attn_matrices=True,
        )
        model.eval()
        _MODEL_CACHE[model_name] = model
        print(f"✅ {model_name} — {model.cfg.n_layers}L {model.cfg.n_heads}H d_model={model.cfg.d_model}")
    return _MODEL_CACHE[model_name]


print("✅ Model session defined")

## 4. Shared Helper: Logit Diff

Most Tier 1 tools measure the **logit difference** between the indirect object token (correct)
and the subject token (incorrect). This is the standard IOI metric from Wang et al. (2022).

In [ ]:
def get_logit_diff(
    model: HookedTransformer,
    tokens: torch.Tensor,
    io_token_ids: list[int],
    subject_token_ids: list[int],
    pos: int = -1,
) -> torch.Tensor:
    """Run model and return mean logit diff (IO logit - subject logit) at position `pos`."""
    with torch.no_grad():
        logits = model(tokens)  # [batch, seq, d_vocab]
    diffs = []
    for i, (io_id, s_id) in enumerate(zip(io_token_ids, subject_token_ids)):
        diff = logits[i, pos, io_id] - logits[i, pos, s_id]
        diffs.append(diff.item())
    return sum(diffs) / len(diffs)


def tokens_to_ids(model: HookedTransformer, token_strs: list[str]) -> list[int]:
    """Convert a list of token strings (with leading space) to vocab IDs."""
    return [model.to_single_token(t) for t in token_strs]


print("✅ Logit diff helper defined")

## 5. Tools

Each tool is a **plain Python function** — no LangGraph, no Claude.
Every tool returns an `ExperimentResult`. Claude fills the `summary` field later.

In [ ]:
# ── Tool 1: Logit Lens ────────────────────────────────────────────────────────
def run_logit_lens(
    model: HookedTransformer,
    prompts: list[str],
    pos: int = -1,
    top_k: int = 5,
    output_dir: Path = PLOTS_DIR,
) -> ExperimentResult:
    """Project residual stream at each layer to vocab space.
    Shows how the model's prediction evolves layer by layer."""

    output_dir.mkdir(parents=True, exist_ok=True)
    tokens   = model.to_tokens(prompts)
    n_layers = model.cfg.n_layers

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    layer_data = []
    for layer in range(n_layers):
        resid = cache["resid_post", layer][:, pos, :]      # [batch, d_model]
        with torch.no_grad():
            logits = model.unembed(model.ln_final(resid))  # [batch, d_vocab]
            probs  = F.softmax(logits, dim=-1)
        top_probs, top_ids = probs[0].topk(top_k)
        layer_data.append({
            "layer":      layer,
            "top_tokens": [model.to_string(t.item()) for t in top_ids],
            "top_probs":  top_probs.tolist(),
        })

    # Heatmap: tokens × layers
    seen, all_tokens = set(), []
    for ld in layer_data:
        for t in ld["top_tokens"]:
            if t not in seen:
                seen.add(t)
                all_tokens.append(t)
    all_tokens = all_tokens[:15]

    prob_matrix = np.zeros((len(all_tokens), n_layers))
    for j, ld in enumerate(layer_data):
        for token, prob in zip(ld["top_tokens"], ld["top_probs"]):
            if token in all_tokens:
                prob_matrix[all_tokens.index(token), j] = prob

    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.imshow(prob_matrix, aspect="auto", cmap="Blues",
                   vmin=0, vmax=max(prob_matrix.max(), 0.01))
    ax.set_xticks(range(n_layers))
    ax.set_xticklabels([str(l) for l in range(n_layers)], fontsize=8)
    ax.set_yticks(range(len(all_tokens)))
    ax.set_yticklabels([repr(t) for t in all_tokens], fontsize=9)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Token")
    ax.set_title(f"Logit Lens — {model.cfg.model_name}\n'{prompts[0][:70]}'")
    plt.colorbar(im, ax=ax, label="Probability")
    plt.tight_layout()

    plot_path = output_dir / "logit_lens.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    return ExperimentResult(
        name="Logit Lens", tool="logit_lens",
        model_name=model.cfg.model_name, prompts=prompts,
        plot_paths=[plot_path],
        data={"layers": layer_data, "position": pos,
              "final_top_token": layer_data[-1]["top_tokens"][0]},
        status="success",
    )


print("✅ run_logit_lens defined")

In [ ]:
# ── Tool 2: Attention Pattern ─────────────────────────────────────────────────
def run_attention_pattern(
    model: HookedTransformer,
    prompts: list[str],
    layers: list[int] | None = None,
    output_dir: Path = PLOTS_DIR,
) -> ExperimentResult:
    """Visualise attention weights per head per layer."""

    output_dir.mkdir(parents=True, exist_ok=True)
    tokens     = model.to_tokens(prompts)
    token_strs = model.to_str_tokens(tokens[0])
    seq_len    = tokens.shape[1]
    n_heads    = model.cfg.n_heads
    n_layers   = model.cfg.n_layers

    if layers is None:
        step = max(1, n_layers // 5)
        layers = list(range(0, n_layers, step))[:5]

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    plot_paths, layer_data = [], {}

    for layer in layers:
        attn = cache["pattern", layer][0]  # [n_heads, seq, seq]
        layer_data[f"layer_{layer}"] = {
            "max_attn_per_head": attn[:, -1, :].max(-1).values.tolist(),
            "final_token_attn":  attn[:, -1, :].tolist(),
        }

        n_cols = min(4, n_heads)
        n_rows = (n_heads + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 2.8))
        axes_flat = np.array(axes).flatten()

        for h in range(n_heads):
            ax = axes_flat[h]
            ax.imshow(attn[h].cpu().numpy(), cmap="Blues", vmin=0, vmax=1)
            ax.set_title(f"L{layer}H{h}", fontsize=8)
            ax.set_xticks(range(seq_len))
            ax.set_xticklabels(token_strs, rotation=90, fontsize=5)
            ax.set_yticks(range(seq_len))
            ax.set_yticklabels(token_strs, fontsize=5)
        for i in range(n_heads, len(axes_flat)):
            axes_flat[i].set_visible(False)

        fig.suptitle(f"Attention Patterns — Layer {layer}", fontsize=10)
        plt.tight_layout()
        plot_path = output_dir / f"attention_layer_{layer}.png"
        fig.savefig(plot_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
        plot_paths.append(plot_path)

    return ExperimentResult(
        name="Attention Pattern", tool="attention_pattern",
        model_name=model.cfg.model_name, prompts=prompts,
        plot_paths=plot_paths,
        data={"layers_analysed": layers, "token_strs": token_strs, **layer_data},
        status="success",
    )


print("✅ run_attention_pattern defined")

In [ ]:
# ── Tool 3: Direct Logit Attribution ─────────────────────────────────────────
def run_direct_logit_attribution(
    model: HookedTransformer,
    prompts: list[str],
    io_tokens: list[str],       # e.g. [" Mary", " Tom"]  — correct answers
    subject_tokens: list[str],  # e.g. [" John", " Sarah"] — incorrect answers
    pos: int = -1,
    output_dir: Path = PLOTS_DIR,
) -> ExperimentResult:
    """Decompose the final logit difference into per-head and per-MLP contributions.
    Uses cache[\'z\', layer] + W_O (always available, no extra flags needed).
    Answers: which components *write* the IO prediction into the residual stream?"""

    output_dir.mkdir(parents=True, exist_ok=True)
    tokens   = model.to_tokens(prompts)
    n_layers = model.cfg.n_layers
    n_heads  = model.cfg.n_heads
    W_U      = model.W_U  # [d_model, d_vocab]

    io_ids = tokens_to_ids(model, io_tokens)
    s_ids  = tokens_to_ids(model, subject_tokens)

    # IO - Subject direction in vocab space
    logit_diff_dir = torch.zeros(model.cfg.d_model)
    for io_id, s_id in zip(io_ids, s_ids):
        logit_diff_dir += W_U[:, io_id] - W_U[:, s_id]
    logit_diff_dir = logit_diff_dir.detach() / len(io_ids)

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    head_attrs = torch.zeros(n_layers, n_heads)
    mlp_attrs  = torch.zeros(n_layers)

    for layer in range(n_layers):
        # cache["z", layer]: [batch, seq, n_heads, d_head]  — always cached
        # model.W_O[layer]:  [n_heads, d_head, d_model]
        z       = cache["z", layer]                                      # [B, S, H, d_head]
        W_O     = model.W_O[layer]                                       # [H, d_head, d_model]
        head_out = torch.einsum("bshd,hdm->bshm", z, W_O)               # [B, S, H, d_model]

        for h in range(n_heads):
            h_vec = head_out[:, pos, h, :].mean(0)       # [d_model], mean over batch
            head_attrs[layer, h] = h_vec @ logit_diff_dir

        # MLP contribution
        mlp_out = cache["mlp_out", layer][:, pos, :].mean(0)            # [d_model]
        mlp_attrs[layer] = mlp_out @ logit_diff_dir

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    vmax = head_attrs.abs().max().item()
    im = axes[0].imshow(head_attrs.detach().numpy(), cmap="RdBu", vmin=-vmax, vmax=vmax)
    axes[0].set_xlabel("Head")
    axes[0].set_ylabel("Layer")
    axes[0].set_title("Head Attribution (IO - Subject logit diff)")
    axes[0].set_xticks(range(n_heads))
    axes[0].set_yticks(range(n_layers))
    plt.colorbar(im, ax=axes[0])

    colors = ["#d73027" if v > 0 else "#4575b4" for v in mlp_attrs.tolist()]
    axes[1].barh(range(n_layers), mlp_attrs.detach().numpy(), color=colors)
    axes[1].set_xlabel("Logit diff contribution")
    axes[1].set_ylabel("Layer")
    axes[1].set_title("MLP Attribution per Layer")
    axes[1].set_yticks(range(n_layers))
    axes[1].axvline(0, color="black", linewidth=0.8)

    plt.suptitle(f"Direct Logit Attribution — {model.cfg.model_name}", fontsize=12)
    plt.tight_layout()
    plot_path = output_dir / "direct_logit_attribution.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    flat      = head_attrs.flatten()
    top_idx   = flat.abs().topk(5).indices
    top_heads = [(int(i // n_heads), int(i % n_heads), flat[i].item()) for i in top_idx]

    return ExperimentResult(
        name="Direct Logit Attribution", tool="direct_logit_attribution",
        model_name=model.cfg.model_name, prompts=prompts,
        plot_paths=[plot_path],
        data={
            "head_attrs":     head_attrs.tolist(),
            "mlp_attrs":      mlp_attrs.tolist(),
            "top_heads":      [(f"L{l}H{h}", round(v, 4)) for l, h, v in top_heads],
            "io_tokens":      io_tokens,
            "subject_tokens": subject_tokens,
        },
        status="success",
    )


print("✅ run_direct_logit_attribution defined")

In [ ]:
# ── Tool 4: Ablation ──────────────────────────────────────────────────────────
def run_ablation(
    model: HookedTransformer,
    prompts: list[str],
    io_tokens: list[str],
    subject_tokens: list[str],
    ablation_type: Literal["zero", "mean"] = "mean",
    output_dir: Path = PLOTS_DIR,
) -> ExperimentResult:
    """Ablate each attention head and measure the drop in logit diff.
    Hooks on hook_z (always available) rather than hook_result.
    Large positive drop = causally important head."""

    output_dir.mkdir(parents=True, exist_ok=True)
    tokens   = model.to_tokens(prompts)
    n_layers = model.cfg.n_layers
    n_heads  = model.cfg.n_heads
    io_ids   = tokens_to_ids(model, io_tokens)
    s_ids    = tokens_to_ids(model, subject_tokens)

    baseline_ld = get_logit_diff(model, tokens, io_ids, s_ids)
    print(f"  Baseline logit diff: {baseline_ld:.4f}")

    # Precompute mean z activations for mean ablation
    mean_z = {}
    if ablation_type == "mean":
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens)
        for layer in range(n_layers):
            # cache["z", layer]: [batch, seq, n_heads, d_head]
            # mean over batch and seq → [1, 1, n_heads, d_head]
            mean_z[layer] = cache["z", layer].mean(dim=[0, 1], keepdim=True)

    ablation_effects = torch.zeros(n_layers, n_heads)

    for layer in range(n_layers):
        for head in range(n_heads):
            if ablation_type == "zero":
                def hook_fn(value, hook, h=head):
                    value = value.clone()
                    value[:, :, h, :] = 0.0   # zero out this head\'s z
                    return value
            else:
                mz = mean_z[layer][:, :, head, :]   # [1, 1, d_head]
                def hook_fn(value, hook, h=head, mz=mz):
                    value = value.clone()
                    value[:, :, h, :] = mz            # replace with mean z
                    return value

            hook_name = f"blocks.{layer}.attn.hook_z"
            with torch.no_grad():
                with model.hooks(fwd_hooks=[(hook_name, hook_fn)]):
                    ablated_ld = get_logit_diff(model, tokens, io_ids, s_ids)

            ablation_effects[layer, head] = baseline_ld - ablated_ld

        if layer % 3 == 0:
            print(f"  Layer {layer}/{n_layers-1} done")

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 6))
    vmax = max(ablation_effects.abs().max().item(), 1e-6)
    im = ax.imshow(ablation_effects.numpy(), cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_xticks(range(n_heads))
    ax.set_yticks(range(n_layers))
    ax.set_title(f"{ablation_type.capitalize()} Ablation — Drop in Logit Diff\n"
                 "Red = important (removing hurts), Blue = suppressive (removing helps)")
    plt.colorbar(im, ax=ax, label="Logit diff drop")
    plt.tight_layout()

    plot_path = output_dir / f"ablation_{ablation_type}.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    flat      = ablation_effects.flatten()
    top_idx   = flat.topk(5).indices
    top_heads = [(int(i // n_heads), int(i % n_heads), flat[i].item()) for i in top_idx]

    return ExperimentResult(
        name=f"Ablation ({ablation_type})", tool="ablation",
        model_name=model.cfg.model_name, prompts=prompts,
        plot_paths=[plot_path],
        data={
            "ablation_type":  ablation_type,
            "baseline_ld":    round(baseline_ld, 4),
            "effects":        ablation_effects.tolist(),
            "top_heads":      [(f"L{l}H{h}", round(v, 4)) for l, h, v in top_heads],
            "io_tokens":      io_tokens,
            "subject_tokens": subject_tokens,
        },
        status="success",
    )


print("✅ run_ablation defined")

In [ ]:
# ── Tool 5: Activation Patching ───────────────────────────────────────────────
def run_activation_patching(
    model: HookedTransformer,
    prompts: list[str],              # clean prompts (model gets right answer)
    corrupted_prompts: list[str],    # corrupted prompts (model gets wrong answer)
    io_tokens: list[str],
    subject_tokens: list[str],
    output_dir: Path = PLOTS_DIR,
) -> ExperimentResult:
    """Activation patching: for each layer × position, patch the residual stream
    from the clean run into the corrupted run and measure how much logit diff recovers.
    High recovery = that layer+position causally carries the IO information."""

    output_dir.mkdir(parents=True, exist_ok=True)
    clean_tokens   = model.to_tokens(prompts)
    corrupt_tokens = model.to_tokens(corrupted_prompts)
    n_layers = model.cfg.n_layers
    seq_len  = clean_tokens.shape[1]
    io_ids   = tokens_to_ids(model, io_tokens)
    s_ids    = tokens_to_ids(model, subject_tokens)

    # Baseline: clean and corrupted logit diffs
    clean_ld   = get_logit_diff(model, clean_tokens,   io_ids, s_ids)
    corrupt_ld = get_logit_diff(model, corrupt_tokens, io_ids, s_ids)
    total_diff = clean_ld - corrupt_ld
    print(f"  Clean LD: {clean_ld:.3f}  |  Corrupt LD: {corrupt_ld:.3f}  |  Delta: {total_diff:.3f}")

    # Cache clean activations
    with torch.no_grad():
        _, clean_cache = model.run_with_cache(clean_tokens)

    # Patch each (layer, position) and record recovery fraction
    patch_effects = torch.zeros(n_layers, seq_len)

    for layer in range(n_layers):
        clean_resid_layer = clean_cache["resid_post", layer]  # [batch, seq, d_model]

        for pos in range(seq_len):
            clean_act = clean_resid_layer[:, pos:pos+1, :].clone()

            def hook_fn(value, hook, ca=clean_act, p=pos):
                value = value.clone()
                value[:, p:p+1, :] = ca
                return value

            with torch.no_grad():
                hook_name = f"blocks.{layer}.hook_resid_post"
                with model.hooks(fwd_hooks=[(hook_name, hook_fn)]):
                    patched_ld = get_logit_diff(model, corrupt_tokens, io_ids, s_ids)

            # Normalised recovery: 0 = no recovery, 1 = full recovery
            if abs(total_diff) > 1e-6:
                patch_effects[layer, pos] = (patched_ld - corrupt_ld) / total_diff

        if layer % 3 == 0:
            print(f"  Layer {layer}/{n_layers-1} done")

    # ── Plot: heatmap layer × position ────────────────────────────────────────
    token_strs = model.to_str_tokens(clean_tokens[0])

    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(patch_effects.numpy(), cmap="RdBu",
                   vmin=-0.5, vmax=1.0, aspect="auto")
    ax.set_xlabel("Token position")
    ax.set_ylabel("Layer")
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(token_strs, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n_layers))
    ax.set_title(f"Activation Patching — Normalised Logit Diff Recovery\n"
                 f"1.0 = full recovery (critical), 0 = no effect")
    plt.colorbar(im, ax=ax, label="Fraction recovered")
    plt.tight_layout()

    plot_path = output_dir / "activation_patching.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Find peak recovery positions
    flat     = patch_effects.flatten()
    top_idx  = flat.topk(5).indices
    top_pos  = [(int(i // seq_len), token_strs[int(i % seq_len)], flat[i].item())
                for i in top_idx]

    return ExperimentResult(
        name="Activation Patching", tool="activation_patching",
        model_name=model.cfg.model_name, prompts=prompts,
        plot_paths=[plot_path],
        data={
            "clean_ld":           round(clean_ld, 4),
            "corrupt_ld":         round(corrupt_ld, 4),
            "patch_effects":      patch_effects.tolist(),
            "token_strs":         token_strs,
            "top_recovery_positions": [(f"L{l} tok='{t}'", round(v, 4)) for l, t, v in top_pos],
            "corrupted_prompts":  corrupted_prompts,
        },
        status="success",
    )


print("✅ run_activation_patching defined")

In [ ]:
# ── Tool Registry ─────────────────────────────────────────────────────────────
TOOL_REGISTRY: dict[str, Any] = {
    "logit_lens":               run_logit_lens,
    "attention_pattern":        run_attention_pattern,
    "direct_logit_attribution": run_direct_logit_attribution,
    "ablation":                 run_ablation,
    "activation_patching":      run_activation_patching,
    # Tier 2 (reasoning models): cot_faithfulness, thinking_token_lens, thought_intervention
    # Tier 3 (SAE): sae_feature_search, feature_steering
    # Tier 4 (safety): linear_probe, representation_reading
}

print(f"✅ Tool registry: {list(TOOL_REGISTRY.keys())}")

## 6. Sandbox

In [ ]:
def run_in_sandbox(
    tool_fn,
    tool_kwargs: dict,
    experiment_name: str,
    timeout: int = 300,   # ablation/patching can take a few minutes on CPU
) -> ExperimentResult:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(tool_fn, **tool_kwargs)
        try:
            return future.result(timeout=timeout)
        except concurrent.futures.TimeoutError:
            return ExperimentResult(
                name=experiment_name, tool="unknown",
                model_name="unknown", prompts=[],
                status="timeout", error=f"Timed out after {timeout}s",
            )
        except Exception as e:
            return ExperimentResult(
                name=experiment_name, tool="unknown",
                model_name="unknown", prompts=[],
                status="failed", error=str(e),
            )


print("✅ Sandbox defined")

## 7. Quick Tool Tests (No LangGraph)

Validate every tool directly on the IOI task before wiring into the workflow.

In [ ]:
# ── IOI dataset ───────────────────────────────────────────────────────────────
IOI_PROMPTS = [
    "When Mary and John went to the store, John gave a drink to",
    "When Tom and Sarah went to the park, Sarah gave a ball to",
    "When Alice and Bob went to the office, Bob gave a pen to",
]
IO_TOKENS      = [" Mary",  " Tom",   " Alice"]  # correct (indirect object)
SUBJECT_TOKENS = [" John",  " Sarah", " Bob"]    # incorrect (subject)

# Corrupted prompts for activation patching: names swapped
IOI_CORRUPTED = [
    "When John and Mary went to the store, Mary gave a drink to",
    "When Sarah and Tom went to the park, Tom gave a ball to",
    "When Bob and Alice went to the office, Alice gave a pen to",
]

model = get_model("gpt2")
print(f"\nBaseline logit diff: {get_logit_diff(model, model.to_tokens(IOI_PROMPTS), tokens_to_ids(model, IO_TOKENS), tokens_to_ids(model, SUBJECT_TOKENS)):.4f}")

In [ ]:
# ── Test 1: Logit Lens ────────────────────────────────────────────────────────
result_ll = run_logit_lens(model, IOI_PROMPTS)
print(f"Status: {result_ll.status}")
print(f"Final layer top token: {result_ll.data['final_top_token']!r}")
from IPython.display import Image, display
display(Image(filename=str(result_ll.plot_paths[0])))

In [ ]:
# ── Test 2: Attention Pattern ─────────────────────────────────────────────────
result_ap = run_attention_pattern(model, IOI_PROMPTS[:1], layers=[0, 5, 9, 10, 11])
print(f"Status: {result_ap.status} | Plots: {len(result_ap.plot_paths)}")
for p in result_ap.plot_paths:
    display(Image(filename=str(p)))

In [ ]:
# ── Test 3: Direct Logit Attribution ─────────────────────────────────────────
result_dla = run_direct_logit_attribution(
    model, IOI_PROMPTS, IO_TOKENS, SUBJECT_TOKENS
)
print(f"Status: {result_dla.status}")
print(f"Top contributing heads: {result_dla.data['top_heads']}")
display(Image(filename=str(result_dla.plot_paths[0])))

In [ ]:
# ── Test 4: Ablation ──────────────────────────────────────────────────────────
# Note: runs n_layers × n_heads forward passes — ~1-2 min on CPU
result_abl = run_ablation(
    model, IOI_PROMPTS, IO_TOKENS, SUBJECT_TOKENS, ablation_type="mean"
)
print(f"Status: {result_abl.status}")
print(f"Baseline LD: {result_abl.data['baseline_ld']}")
print(f"Most important heads: {result_abl.data['top_heads']}")
display(Image(filename=str(result_abl.plot_paths[0])))

In [ ]:
# ── Test 5: Activation Patching ───────────────────────────────────────────────
# Note: runs n_layers × seq_len forward passes — ~2-3 min on CPU
result_ap2 = run_activation_patching(
    model, IOI_PROMPTS, IOI_CORRUPTED, IO_TOKENS, SUBJECT_TOKENS
)
print(f"Status: {result_ap2.status}")
print(f"Clean LD: {result_ap2.data['clean_ld']} | Corrupt LD: {result_ap2.data['corrupt_ld']}")
print(f"Top recovery positions: {result_ap2.data['top_recovery_positions']}")
display(Image(filename=str(result_ap2.plot_paths[0])))

## 8. LangGraph Workflow

```
parse_plan → load_model → run_experiment → interpret_result → check_followup
                               ▲                                      │
                               └──────── more experiments? ───────────┘
                                                                       │
                                                               collect_results → END
```

In [ ]:
# ── State ─────────────────────────────────────────────────────────────────────
class LensState(TypedDict):
    research_plan:     str
    research_question: str
    experiment_queue:  list[dict]
    model_name:        str
    last_result:       dict | None
    followup_count:    int
    results:           list[dict]
    bundle:            dict | None


class ExperimentSpecModel(BaseModel):
    name: str
    tool: str
    model_name: str
    prompts: list[str]
    what_to_measure: str
    hypothesis_tested: str
    expected_outcome: str
    tool_kwargs: dict = {}


class ParsedPlanModel(BaseModel):
    research_question: str
    model_name: str
    experiments: list[ExperimentSpecModel]


class FollowUpDecision(BaseModel):
    needs_followup: bool
    reason: str
    followup_tool: str | None = None
    followup_prompts: list[str] = []
    followup_description: str | None = None
    followup_kwargs: dict = {}


print("✅ State and Pydantic models defined")

In [ ]:
# ── Node 1: parse_plan ────────────────────────────────────────────────────────
def parse_plan(state: LensState) -> dict:
    print("📋 [parse_plan] Extracting experiments...")
    structured_llm = llm.with_structured_output(ParsedPlanModel)
    prompt = f"""Extract the experiments from this Research Plan.
For each experiment, extract the tool name, model, prompts, and what to measure.
Only include experiments whose tool is one of: {list(TOOL_REGISTRY.keys())}.
For tools that need io_tokens and subject_tokens, include them in tool_kwargs.
If no specific prompts are given, generate 2-3 appropriate IOI-style prompts.

Research Plan:
{state['research_plan']}"""

    parsed = structured_llm.invoke(prompt)
    queue  = [e.model_dump() for e in parsed.experiments]
    print(f"   Found {len(queue)} experiments: {[e['name'] for e in queue]}")
    return {
        "research_question": parsed.research_question,
        "model_name":        parsed.model_name,
        "experiment_queue":  queue,
        "results":           [],
        "followup_count":    0,
        "last_result":       None,
        "bundle":            None,
    }


# ── Node 2: load_model ────────────────────────────────────────────────────────
def load_model_node(state: LensState) -> dict:
    print(f"🧠 [load_model] {state['model_name']}")
    get_model(state["model_name"])
    return {}


print("✅ parse_plan, load_model defined")

In [ ]:
# ── Node 3: run_experiment ────────────────────────────────────────────────────
def run_experiment(state: LensState) -> dict:
    queue = list(state["experiment_queue"])
    spec  = queue.pop(0)
    print(f"\n🔬 [run_experiment] '{spec['name']}' using {spec['tool']}")

    tool_fn = TOOL_REGISTRY.get(spec["tool"])
    if tool_fn is None:
        result = ExperimentResult(
            name=spec["name"], tool=spec["tool"],
            model_name=spec["model_name"], prompts=spec["prompts"],
            status="failed",
            error=f"Tool '{spec['tool']}' not in registry: {list(TOOL_REGISTRY.keys())}",
        )
    else:
        m = get_model(spec["model_name"])
        extra_kwargs = spec.get("tool_kwargs", {})
        result = run_in_sandbox(
            tool_fn,
            tool_kwargs={"model": m, "prompts": spec["prompts"], **extra_kwargs},
            experiment_name=spec["name"],
        )

    icon = "✅" if result.status == "success" else "❌"
    print(f"   {icon} {result.status} | plots={len(result.plot_paths)}")
    return {
        "experiment_queue": queue,
        "last_result": result.__dict__ | {"plot_paths": [str(p) for p in result.plot_paths]},
    }


print("✅ run_experiment defined")

In [ ]:
# ── Node 4: interpret_result ──────────────────────────────────────────────────
def interpret_result(state: LensState) -> dict:
    result = state["last_result"]
    print(f"💬 [interpret_result] '{result['name']}'")

    if result["status"] != "success":
        result["summary"] = f"Experiment failed: {result.get('error')}"
        return {"results": list(state["results"]) + [result], "last_result": result}

    data_preview = json.dumps(result["data"], indent=2)[:2_000]
    response = llm.invoke(f"""You are an AI safety researcher specialised in mechanistic interpretability.
Interpret this experiment result in 2-3 paragraphs:
1. What do the results reveal about the model's internal mechanisms?
2. What is the most important finding?
3. How does this connect to the research question: "{state['research_question']}"

Experiment: {result['name']} | Tool: {result['tool']}
Data: {data_preview}""")

    result["summary"] = response.content
    return {"results": list(state["results"]) + [result], "last_result": result}


# ── Node 5: check_followup ────────────────────────────────────────────────────
def check_followup(state: LensState) -> dict:
    result = state["last_result"]
    followup_count = state.get("followup_count", 0)
    if followup_count >= 2 or result["status"] != "success":
        return {}

    structured_llm = llm.with_structured_output(FollowUpDecision)
    decision = structured_llm.invoke(f"""Based on this result, decide if an immediate follow-up is needed.
Be conservative — only if the result reveals something a different tool can directly test.
Available tools: {list(TOOL_REGISTRY.keys())}
Follow-ups used: {followup_count}/2
Summary: {result['summary']}""")

    if decision.needs_followup and decision.followup_tool:
        new_spec = {
            "name": f"Follow-up: {decision.followup_description or decision.followup_tool}",
            "tool": decision.followup_tool,
            "model_name": result["model_name"],
            "prompts": decision.followup_prompts or result["prompts"],
            "what_to_measure": decision.followup_description or "",
            "hypothesis_tested": "follow-up",
            "expected_outcome": "",
            "tool_kwargs": decision.followup_kwargs,
        }
        print(f"   ➕ Follow-up: {new_spec['name']}")
        return {
            "experiment_queue": [new_spec] + list(state["experiment_queue"]),
            "followup_count": followup_count + 1,
        }
    return {}


# ── Node 6: collect_results ───────────────────────────────────────────────────
def collect_results(state: LensState) -> dict:
    print("\n📦 [collect_results] Bundling...")
    bundle = {
        "research_question": state["research_question"],
        "model_name":        state["model_name"],
        "results":           state["results"],
        "n_total":           len(state["results"]),
        "n_success":         sum(1 for r in state["results"] if r["status"] == "success"),
        "n_failed":          sum(1 for r in state["results"] if r["status"] != "success"),
    }
    bundle_path = OUTPUTS_DIR / "results_bundle.json"
    bundle_path.write_text(json.dumps(bundle, indent=2, default=str))
    print(f"   ✅ {bundle['n_success']}/{bundle['n_total']} succeeded → {bundle_path}")
    return {"bundle": bundle}


def route_after_followup(state: LensState) -> str:
    return "run_experiment" if state.get("experiment_queue") else "collect_results"


print("✅ All nodes defined")

In [ ]:
# ── Build the graph ───────────────────────────────────────────────────────────
workflow = StateGraph(LensState)
workflow.add_node("parse_plan",       parse_plan)
workflow.add_node("load_model",       load_model_node)
workflow.add_node("run_experiment",   run_experiment)
workflow.add_node("interpret_result", interpret_result)
workflow.add_node("check_followup",   check_followup)
workflow.add_node("collect_results",  collect_results)

workflow.set_entry_point("parse_plan")
workflow.add_edge("parse_plan",       "load_model")
workflow.add_edge("load_model",       "run_experiment")
workflow.add_edge("run_experiment",   "interpret_result")
workflow.add_edge("interpret_result", "check_followup")
workflow.add_conditional_edges(
    "check_followup", route_after_followup,
    {"run_experiment": "run_experiment", "collect_results": "collect_results"},
)
workflow.add_edge("collect_results", END)

memory = MemorySaver()
lens   = workflow.compile(checkpointer=memory)
print("✅ Lens compiled:", list(workflow.nodes.keys()))

## 9. Run Lens

In [ ]:
SAMPLE_RESEARCH_PLAN = """
# Research Plan: IOI Circuit in GPT-2 Small

## Research Question
How does GPT-2 Small implement the Indirect Object Identification (IOI) task?

## Target Model
gpt2

## Proposed Experiments

### Experiment 1: Logit Lens
- Lens Tool: logit_lens
- Model: gpt2
- Prompts: ["When Mary and John went to the store, John gave a drink to", "When Tom and Sarah went to the park, Sarah gave a ball to"]
- What to measure: Layer at which the IO token becomes the top prediction

### Experiment 2: Direct Logit Attribution
- Lens Tool: direct_logit_attribution
- Model: gpt2
- Prompts: ["When Mary and John went to the store, John gave a drink to", "When Tom and Sarah went to the park, Sarah gave a ball to"]
- tool_kwargs: {"io_tokens": [" Mary", " Tom"], "subject_tokens": [" John", " Sarah"]}
- What to measure: Which heads contribute most to the IO logit

### Experiment 3: Ablation
- Lens Tool: ablation
- Model: gpt2
- Prompts: ["When Mary and John went to the store, John gave a drink to", "When Tom and Sarah went to the park, Sarah gave a ball to"]
- tool_kwargs: {"io_tokens": [" Mary", " Tom"], "subject_tokens": [" John", " Sarah"]}
- What to measure: Which heads are causally necessary
""".strip()

In [ ]:
from datetime import datetime
THREAD_ID = f"lens-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
config    = {"configurable": {"thread_id": THREAD_ID}}

print(f"🔬 Running Lens — {THREAD_ID}")
print("=" * 60)

final_state = lens.invoke({"research_plan": SAMPLE_RESEARCH_PLAN}, config=config)

print("\n" + "=" * 60)
print("✅ Complete")

## 10. Display Results

In [ ]:
from IPython.display import Markdown, display, Image

bundle  = final_state.get("bundle", {})
results = bundle.get("results", [])

print(f"Question  : {bundle.get('research_question')}")
print(f"Model     : {bundle.get('model_name')}")
print(f"Succeeded : {bundle.get('n_success')}/{bundle.get('n_total')}")

for r in results:
    icon = "✅" if r["status"] == "success" else "❌"
    display(Markdown(f"### {icon} {r['name']} (`{r['tool']}`)"))
    display(Markdown(r.get("summary", "*No summary.*")))
    for plot_path in r.get("plot_paths", []):
        p = Path(plot_path)
        if p.exists():
            display(Image(filename=str(p)))

## 11. Next Steps

### Tier 2 — Reasoning Model Tools
- [ ] `run_cot_faithfulness` — compare behaviour with/without CoT (nnsight)
- [ ] `run_thinking_token_lens` — logit lens during `<think>` tokens
- [ ] `run_thought_intervention` — corrupt specific reasoning steps, measure answer change

### Tier 3 — SAE / Feature Tools
- [ ] `run_sae_feature_search` — which SAELens features activate for this input?
- [ ] `run_feature_steering` — amplify a feature direction, measure behaviour change

### Tier 4 — Safety-Specific
- [ ] `run_linear_probe` — does model represent concept X in hidden states?
- [ ] `run_representation_reading` — RepE-style belief/intent readout

### Architecture
- [ ] Move tools to `lens/mcp_server/src/tools/` following Nova's pattern
- [ ] Add `config/settings.py` — device, timeout, model config
- [ ] Replace ThreadPoolExecutor sandbox with Modal for GPU isolation
- [ ] Wire Scout output → Lens input in the orchestrator